# S26 · Good and bad charts

We put a misleading chart next to an honest chart of the *same* data. You will see
that the trick is in the chart, not in the numbers. Then we draw a fan chart. It
shows uncertainty honestly, instead of pretending we know the future exactly. We
only use `matplotlib` and `numpy`.

**New here? Read this once.**

- New to Python? You can run this whole notebook. Press play on each cell, from top
  to bottom, and read the short note above each one. What matters is what the
  pictures show, not the code.
- There is no new maths today. If you have learnt about honest charts before, this
  will feel familiar. If not, `primers/choosing_a_chart.md` explains it with
  pictures in about ten minutes.
- Already confident with code? Look for the cell marked **Stretch (optional)** near
  the end.
- Don't know a word? Look in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook only uses matplotlib and numpy, which Colab already ships,
# so there is nothing to install here.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np                       # fast maths on lists of numbers
import matplotlib.pyplot as plt           # drawing every chart in this notebook

## Step 1: the data (it never changes)

Here are four quarters of sales, in Rs lakh. We will show this *one* set of data in
an honest way and in a misleading way. The numbers stay the same; only the chart
changes.

In [ ]:
# Set a seed so the random series later is the same every run.
np.random.seed(0)

quarters = ["Q1", "Q2", "Q3", "Q4"]
sales    = [101.0, 102.0, 103.0, 104.5]    # sales in Rs lakh

# The real change from Q1 to Q4, as a percentage.
percent_change = (sales[3] - sales[0]) / sales[0] * 100
print("Sales (Rs lakh):", sales)
print("Real change Q1 to Q4:", round(percent_change, 1), "%")

## Step 2: the misleading chart (cut-off y-axis)

This is a very common trick. The value axis starts high (here at 100) instead of
at 0. This is called a truncated (cut-off) axis. A small 3.5% rise then looks like
a huge jump. The numbers are correct, but the picture is not honest.

In [ ]:
plt.figure(figsize=(6, 5))
plt.bar(quarters, sales, color="#C0392B", width=0.6)

# The dishonest part: the axis starts at 100, not 0.
plt.ylim(100, 105)

plt.ylabel("sales (Rs lakh)")
plt.title("Misleading: y-axis starts at 100  ->  'sales exploded!'")
plt.show()

## Step 3: the honest chart (same data, axis from 0)

These are exactly the same four numbers. When the axis starts at 0, we see the
truth: a small, steady rise. A bar chart should almost always start at zero.

In [ ]:
plt.figure(figsize=(6, 5))
plt.bar(quarters, sales, color="#2E9E5B", width=0.6)

# The honest fix: start the axis at 0.
plt.ylim(0, 120)

plt.ylabel("sales (Rs lakh)")
plt.title("Honest: y-axis starts at 0  ->  'sales up ~3.5%'")
plt.show()

## Step 4: side by side, so you can see the difference

`plt.subplots` puts two charts in one figure. The data on the left and the right
is the same; only the y-axis range is different. Remember this picture, and look
for this trick in other people's slides.

In [ ]:
# 1 row, 2 columns of charts. Only the y-axis range will differ.
fig, (left_chart, right_chart) = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: the misleading version.
left_chart.bar(quarters, sales, color="#C0392B", width=0.6)
left_chart.set_ylim(100, 105)
left_chart.set_title("Misleading (axis at 100)")
left_chart.set_ylabel("sales (Rs lakh)")

# Right: the honest version.
right_chart.bar(quarters, sales, color="#2E9E5B", width=0.6)
right_chart.set_ylim(0, 120)
right_chart.set_title("Honest (axis at 0)")

fig.suptitle("Same four numbers, two very different stories")
plt.show()

## Step 5: a second trick, cherry-picking

Cherry-picking means showing only the part of the data that looks good. We make
three years of monthly numbers that are mostly flat, with one lucky rise in the
middle. If we show only the rise, the story sounds like a big success. If we show
all the months, we see the truth.

In [ ]:
# Three years of monthly numbers: mostly flat, moving up and down at random.
months = np.arange(0, 36)
series = 100 + np.cumsum(np.random.normal(0.05, 1.4, size=36))

# Add a rise that looks good, in months 18..23.
series[18:24] = series[18:24] + np.linspace(0, 7, 6)

print("We have", len(series), "months of data.")
print("Months 18-23 have a lucky rise.")

## Step 6: the chosen 6 months vs all 36 months

Left: only the 6 months that look good, so the line climbs steeply. Right: all 36
months, with the same 6 months shaded. Now the rise is only a small bump. Always
show enough months so that people can judge the trend fairly.

In [ ]:
fig, (cherry, full) = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: only the 6 months that look good.
cherry.plot(months[18:24], series[18:24], color="#C0392B", linewidth=3)
cherry.set_title("Cherry-picked 6 months")
cherry.set_xlabel("month")
cherry.set_ylabel("value")

# Right: all the months, with the chosen 6 months shaded.
full.plot(months, series, color="#2E75B6", linewidth=2)
full.axvspan(18, 23, color="#F2C14E", alpha=0.3)   # shade the chosen months
full.set_title("Full 3-year series (context restored)")
full.set_xlabel("month")

fig.suptitle("Cherry-picked windows hide the real trend")
plt.show()

## Step 7: showing uncertainty, one number vs a fan chart

A forecast is never one certain number. We have actual revenue for months 0 to 4,
and we forecast months 4 to 6. First we set up three things: the actual values,
the most likely forecast, and how wide the uncertainty is in each future month.

In [ ]:
# Actual revenue we have already seen, months 0..4 (Rs lakh).
actual_months = np.array([0, 1, 2, 3, 4])
actual_values = np.array([100.0, 103.0, 107.0, 110.0, 114.0])

# Forecast for months 4..6 (start at the last actual so the lines join up).
forecast_months = np.array([4, 5, 6])
most_likely     = np.array([114.0, 118.0, 122.0])   # the central guess
half_range      = np.array([0.0, 4.0, 9.0])          # uncertainty grows with time

print("Most-likely forecast for month 6:", most_likely[-1], "Rs lakh")
print("But the range there is", most_likely[-1] - 2 * half_range[-1],
      "to", most_likely[-1] + 2 * half_range[-1], "Rs lakh")

## Step 8: one number only (false certainty)

A single dashed line says "it will be 122". It looks confident, but the real value
will almost never be exactly 122. Do NOT present a forecast like this.

In [ ]:
plt.figure(figsize=(6.5, 5))

# The actuals.
plt.plot(actual_months, actual_values, "-o", color="#2E75B6", linewidth=2, label="actual")

# The forecast as one confident line.
plt.plot(forecast_months, most_likely, "--o", color="#C0392B", linewidth=2, label="forecast")

plt.annotate("'It will be 122.'", xy=(6, 122), xytext=(3.0, 128),
             color="#C0392B",
             arrowprops=dict(arrowstyle="->", color="#C0392B"))

plt.xlabel("month")
plt.ylabel("revenue (Rs lakh)")
plt.title("One number only: looks more certain than it is")
plt.ylim(95, 140)
plt.legend()
plt.show()

## Step 9: the fan chart (honest about uncertainty)

`plt.fill_between` shades a band between a low line and a high line. We draw a
wide band (the full range of outcomes) and a narrow band (the most likely zone).
The reader can see the uncertainty grow into the future.

So instead of "it will be 122", we now give a likely range. For month 6: most
likely about Rs 122 lakh, and very likely between Rs 104 lakh and Rs 140 lakh
(the printed range from Step 7).

In [ ]:
plt.figure(figsize=(6.5, 5))

# The actuals, same as before.
plt.plot(actual_months, actual_values, "-o", color="#2E75B6", linewidth=2, label="actual")

# Wide band: most_likely plus/minus 2 * half_range (the full range of outcomes).
plt.fill_between(forecast_months,
                 most_likely - 2 * half_range,
                 most_likely + 2 * half_range,
                 color="#9DC3E6", alpha=0.5, label="range of outcomes")

# Narrow band: most_likely plus/minus 1 * half_range (the most-likely zone).
plt.fill_between(forecast_months,
                 most_likely - half_range,
                 most_likely + half_range,
                 color="#2E75B6", alpha=0.4, label="most likely")

# The central path as a dashed line through the middle.
plt.plot(forecast_months, most_likely, "--", color="#1F4E79", linewidth=2)

plt.annotate("'most likely about 122,\nlikely 104 to 140'", xy=(6, 122), xytext=(2.7, 132),
             color="#1F4E79",
             arrowprops=dict(arrowstyle="->", color="#1F4E79"))

plt.xlabel("month")
plt.ylabel("revenue (Rs lakh)")
plt.title("Fan chart: honest about uncertainty")
plt.ylim(95, 140)
plt.legend(loc="lower right")
plt.show()

### Stretch (optional): why the band gets wider

Skip this if you only want to present well. The band gets wider each month because
uncertainty adds up: the further ahead you forecast, the less sure you are.

The wide band here is the central guess plus or minus two "half-ranges". Think of
the spread as a bell-shaped distribution. Then plus or minus two standard
deviations covers about 95% of likely outcomes, so the wide band is roughly a "95%
range". The cell below prints the full width of this band in each future month, so
you can see it grow.

In [ ]:
# Full width of the wide band = 2 * half_range on each side = 4 * half_range total.
for month, spread in zip(forecast_months, half_range):
    full_width = 4 * spread
    print("month", month, ": full range width =", round(full_width, 1), "Rs lakh")

## What you did

You saw how the *same* numbers can mislead, with a cut-off axis or cherry-picking,
and how to redraw them honestly. You also changed a single forecast number into a
fan chart that is honest about uncertainty. Honest charts are the right thing to
do. They also protect you when the real result is a little different from your
central guess. Use these habits in the lab when you redraw a chart from one of
your own projects.